# Flow matching for a handwritten digits dataset

In [ ]:
%load_ext autoreload
%autoreload 2

import models  # Auxiliary module for neural network architecture and training
import plotting  # Auxiliary module for visualizations
import data # Auxiliary module for data generation and loading
import generators # Auxiliary module for generating new samples using a flow model

## Load data

We load a handwritten digits dataset:

In [ ]:
DATASET = "mnist" # Options: "digits", "mnist", "cats"
MODEL_FILE_NAME = f"{DATASET}.pt" # File name for saving/loading the trained model

In [ ]:
if DATASET == "digits":
    target_dataset, labels_set = data.load_digits()
elif DATASET == "mnist":
    target_dataset, labels_set = data.load_mnist()
elif DATASET == "cats":
    target_dataset, labels_set = data.load_cats(image_size=64)

Here is a sample of the images we will try to replicate:

In [ ]:
plotting.plot_image_grid(target_dataset)

## Train rectified flow

We start by generating sample from a gaussian distribution with the same shape as the images we want to generate. We also create independent couplings between the source and target distributions.

In [ ]:
num_couplings = 100000
couplings_dataset = data.IndependentDistributionsCouplingsDataset(target_dataset, num_couplings, labels_set=labels_set)

Let's now traing a rectified flow:

In [ ]:
network = models.train_flow_model(couplings_dataset, network="unet", network_args={"num_blocks": 4}, num_epochs=100, verbose=True, log_frequency=1, batch_size=32)

Show architecture of trained network

In [ ]:
plotting.plot_network(network, couplings_dataset, save_filename="network_architecture", dpi="150")  # Render to file in higher quality
plotting.plot_network(network, couplings_dataset, dpi="70")

Save to disk the trained network

In [ ]:
import torch
output_path = f"models/{MODEL_FILE_NAME}"
torch.save(network, output_path)

Once the flow is trained, we can generate some samples:

In [ ]:
synthetic_samples = generators.FlowOutputsDataset(
    network, 
    shape=target_dataset[0][0].shape if labels_set is not None else target_dataset[0].shape,
    num_couplings=1000, 
    labels_set=labels_set, 
    simulation_arguments={"n_steps": 200}
)

And we can also visualize the generated images

In [ ]:
from torch.utils.data import DataLoader
from data import couplings_collate_fn

n_samples = 1000
dataloader = DataLoader(synthetic_samples, batch_size=n_samples, collate_fn=couplings_collate_fn)
samplings_batch = next(iter(dataloader))

# Show synthetic images obtained
images = [(samplings_batch[0][i], int(samplings_batch[1][i])) for i in range(n_samples)] if labels_set is not None else samplings_batch
plotting.plot_image_grid(images)

Visualize the generated trajectories in pixel space

In [ ]:
import numpy as np
samples = 100
source_data = np.random.randn(*([samples] + list(target_dataset[0].shape if isinstance(target_dataset, torch.Tensor) else target_dataset[0][0].shape)))
source_labels = [list(labels_set)[i % len(labels_set)] for i in range(samples)] if labels_set is not None else None
trajectories = models.compute_trajectories(network, source_data, labels=source_labels)

In [ ]:
animation = plotting.animate_image_trajectories(trajectories, source_labels)
animation